# PLM (Hugging Face) vs LLM

LLM과 PLM은 둘 다 자연어 처리를 위한 언어 모델이지만, 그 목적과 규모, 그리고 활용 방식에서 차이가 있음.

1. LLM (Large Language Model):

- 규모: 

수십억에서 수천억 개의 매개변수를 가진 대규모 언어 모델을 의미함.

- 기능:

 광범위한 텍스트 생성, 번역, 요약, 질문 답변 등 다양한 작업을 수행할 수 있음.

- 예시:

 GPT-3, GPT-4 등 OpenAI에서 개발한 모델들이 대표적인 LLM임.

- 특징:

 방대한 양의 데이터를 기반으로 훈련되어 일반적인 지식을 많이 포함하고 있으며, 별도의 미세 조정 없이도 다양한 작업에 활용될 수 있음.


2. PLM (Pretrained Language Model):

- 훈련 방식: 

대량의 텍스트 데이터를 기반으로 사전 훈련된 언어 모델을 의미함.

- 목적: 

특정 작업에 맞게 미세 조정(fine-tuning)하여 성능을 향상시키는 것이 일반적임.

- 예시: 

BERT, RoBERTa, ALBERT 등은 대표적인 PLM임.

- 특징: 

주로 문장의 이해나 분류, 개체명 인식 등 특정 자연어 처리 작업에 사용됨.

> 차이점 요약:

* 규모와 범용성: LLM은 매우 큰 규모로 일반적인 용도로 사용될 수 있지만, PLM은 특정 작업에 맞게 미세 조정되는 경우가 많음.
* 활용 방식: LLM은 별도의 미세 조정 없이도 다양한 작업을 수행할 수 있는 반면, PLM은 미세 조정을 통해 특정 작업의 성능을 최적화함.
* 모델 구조: LLM은 주로 생성 모델에 속하며, PLM은 인코더 또는 디코더 구조를 가질 수 있음.

따라서, LLM은 대규모 데이터와 매개변수를 활용하여 범용적인 언어 이해와 생성을 목표로 하며, PLM은 사전 훈련된 모델을 기반으로 특정 작업에 최적화하는 데 중점을 둠.


# BertForMaskedLM을 이용한 [MASK] 예측
- KLUE 데이터를 기반으로 훈련된 BERT 모델을 허깅페이스에서 다운 받고, 토큰을 가린 문장에 대해 잘 예측하는지 확인

In [1]:
import torch
from transformers import BertTokenizer, BertModel, BertForMaskedLM


model_name = "klue/bert-base"

/Users/khb43/anaconda3/envs/dl_lecture-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Tokenizer 설정 및 확인

In [2]:
tokenizer = BertTokenizer.from_pretrained(model_name)

항상 제일 앞에는 CLS 토큰, 각 문장 끝에는 SEP 토큰이 들어가야 한다. 그래야 Bert에 들어갈 수 있다.

In [3]:
text = "[CLS]대한민국의 수도는 서울입니다. [SEP] 서울의 대표적인 관광지는 남산타워 입니다. [SEP]"

tokenized_text = tokenizer.tokenize(text)

print(tokenized_text)

['[CLS]', '대한민국', '##의', '수도', '##는', '서울', '##입니다', '.', '[SEP]', '서울', '##의', '대표', '##적인', '관광지', '##는', '남산', '##타', '##워', '입니다', '.', '[SEP]']


MASKING을 해보자!

In [4]:
# 가릴 부분 지정: 남산을 마스킹 해보자
masked_index = 15 
print(tokenized_text[masked_index])

남산


In [5]:
# MASK 적용
tokenized_text[masked_index] = "[MASK]"
print(tokenized_text)

['[CLS]', '대한민국', '##의', '수도', '##는', '서울', '##입니다', '.', '[SEP]', '서울', '##의', '대표', '##적인', '관광지', '##는', '[MASK]', '##타', '##워', '입니다', '.', '[SEP]']


In [6]:
# 정수 인코딩 
indexed_tokens = tokenizer.convert_tokens_to_ids(tokenized_text)
print(indexed_tokens)

[2, 4892, 2079, 4438, 2259, 3671, 12190, 18, 3, 3671, 2079, 3661, 31221, 9417, 2259, 4, 2256, 2667, 3714, 18, 3]


위에서 보았듯이 가장 중요한 건 항상 제일 앞에는 CLS 토큰, 각 문장 끝에는 SEP 토큰이 들어가듯이

각 문장을 나눠 넣는 SEGMENT EMBEDDING이다.

In [7]:
# segment_ids 만들기
def get_segment_ids(tokenized_text):

    segment_ids = []
    current_segment_id = 0
    for token in tokenized_text:
        segment_ids.append(current_segment_id)
        if token == '[SEP]':
            current_segment_id += 1

    return segment_ids


In [8]:
segment_ids = get_segment_ids(tokenized_text)
print(segment_ids)

[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [9]:
# Pytorch Tensor 생성

tokens_tensor = torch.tensor([indexed_tokens])
segments_tensor = torch.tensor([segment_ids])

print(tokens_tensor)
print(segments_tensor)

tensor([[    2,  4892,  2079,  4438,  2259,  3671, 12190,    18,     3,  3671,
          2079,  3661, 31221,  9417,  2259,     4,  2256,  2667,  3714,    18,
             3]])
tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


## 모델 다운로드

In [10]:
# [MASK] 토큰으로 마스킹된 부분을 예측할 수 있는 모델 불러오기
model = BertForMaskedLM.from_pretrained(model_name) # 모델과 토크나이저의 모델명은 동일해야 한다.
model

BertForMaskedLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.
Some weights of the model checkpoint at klue/bert-base were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.

BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(32000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwi

In [11]:
# 파이토치를 이용해 모델 객체를 만들고 나서 어떤 장치(device) 환경에서 훈련이나 추론을 수행할지 결정지어주기

import torch

# MPS 지원 여부 확인
if torch.backends.mps.is_available():
    device = 'mps'  # Apple Silicon에서 MPS 사용
elif torch.cuda.is_available():
    device = 'cuda'  # CUDA 사용 가능 시
else:
    device = 'cpu'  # 그 외에는 CPU 사용

device

print(f"Using device: {device}")

Using device: mps


In [12]:
# GPU 세팅. 모델과 입력 데이터 텐서 모두 gpu로 옮겨주기

tokens_tensor = tokens_tensor.to(device)
segments_tensor = segments_tensor.to(device)
model.to(device)

BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(32000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwi

In [13]:
# 모델을 예측 모드로 변경
model.eval()

with torch.no_grad():
    outputs = model(tokens_tensor, token_type_ids=segments_tensor) # outputs에는 모델의 예측 결과가 저장
    predictions = outputs[0] # outputs[0]은 예측 결과의 첫 번째 요소로, 모든 단어에 대한 예측 점수를 포함한 텐서(각 위치에서 가능한 모든 단어에 대한 점수)
    # 이를 통해 마스킹된 위치의 단어를 예측하는 데 필요한 정보를 얻음.
predictions.shape

torch.Size([1, 21, 32000])

In [14]:
predicted_index = torch.argmax(predictions[0, masked_index]).item() 
# 모델이 가장 가능성이 높다고 판단하는 단어의 인덱스
# item()을 통해 이 인덱스를 파이썬 정수로 반환

predicted_index

15555

In [15]:
predicted_token = tokenizer.convert_ids_to_tokens([predicted_index])[0]
# convert_ids_to_tokens 함수는 인덱스 값을 실제 단어 토큰으로 변환 
# 최종적으로, predicted_token에는 모델이 마스킹된 위치에 예측한 단어가 저장

predicted_token

'롯데월드'

## 파이프라인을 이용한 예측
대부분의 자연어로 할 수 있는 처리들을 파이프라인화 하여 입력 부터 예측까지 하나의 단계로 완성할 수 있게 해 준다.

In [16]:
from transformers import pipeline

# 트랜스포머 모델을 이용해서 어떤 작업(task)을 수행할지 지정하고, 해당하는 모델을 넣어주면 된다.
pipe = pipeline('fill-mask', model=model, tokenizer=tokenizer, device='mps')  # MPS를 사용하도록 설정

result = pipe("부산광역시는 대표적인 한국의 도시입니다. 부산의 대표적인 관광지는 [MASK] 입니다.")
result

[{'score': 0.5292262434959412,
  'token': 9568,
  'token_str': '해운대',
  'sequence': '부산광역시는 대표적인 한국의 도시입니다. 부산의 대표적인 관광지는 해운대 입니다.'},
 {'score': 0.0649067759513855,
  'token': 11615,
  'token_str': '해수욕장',
  'sequence': '부산광역시는 대표적인 한국의 도시입니다. 부산의 대표적인 관광지는 해수욕장 입니다.'},
 {'score': 0.0440373569726944,
  'token': 3902,
  'token_str': '부산',
  'sequence': '부산광역시는 대표적인 한국의 도시입니다. 부산의 대표적인 관광지는 부산 입니다.'},
 {'score': 0.04394267126917839,
  'token': 10509,
  'token_str': '온천',
  'sequence': '부산광역시는 대표적인 한국의 도시입니다. 부산의 대표적인 관광지는 온천 입니다.'},
 {'score': 0.024953249841928482,
  'token': 15879,
  'token_str': '케이블카',
  'sequence': '부산광역시는 대표적인 한국의 도시입니다. 부산의 대표적인 관광지는 케이블카 입니다.'}]

# RoBERTa를 이용한 텍스트 분류
- NLI

In [17]:
from transformers import pipeline, AutoTokenizer

classifier = pipeline(
    'text-classification',
    model = "Huffon/klue-roberta-base-nli",
    return_all_scores=True
)

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.
/Users/khb43/anaconda3/envs/dl_lecture-env/lib/python3.11/site-packages/transformers/pipelines/text_classification.py:104: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [18]:
tokenizer = AutoTokenizer.from_pretrained("Huffon/klue-roberta-base-nli")

In [19]:
# ENTAILMENT : 논리적인 문장 구조
# NEUTRAL : 중립적인 문장 구조
# CONTRADICTION : 모순적인 문장 구조

classifier(
    f"나는 악기를 연주하는 것을 좋아한다. {tokenizer.sep_token} 나는 악기를 다루는 것이 싫다."
)

[[{'label': 'ENTAILMENT', 'score': 0.0003440291038714349},
  {'label': 'NEUTRAL', 'score': 0.00041993800550699234},
  {'label': 'CONTRADICTION', 'score': 0.999235987663269}]]